# 08 — Variáveis não exploradas do SCR

O pipeline usa **7 das 23 colunas** do SCR. O diagnóstico dos notebooks anteriores mostrou por que
isso limita o resultado: o conjunto de variáveis não contém **nenhuma informação exógena que varie
simultaneamente por segmento e no tempo**. As defasagens derivam todas da mesma série do alvo; as
macro são idênticas para os 494 segmentos no mesmo mês; as categóricas são constantes no tempo.

Este notebook incorpora colunas do SCR que **já estão baixadas** e preenchem exatamente essa lacuna.

## Variáveis incorporadas

| Coluna do SCR | Por que importa |
|---|---|
| `ativo_problematico` | Medida **mais ampla** que a carteira inadimplida (Res. CMN 4.557): inclui operações reestruturadas e com sinais de deterioração ainda não vencidas. É um indicador antecedente por construção. |
| `a_vencer_*` (6 faixas) | **Estrutura de maturidade** da carteira. Um segmento com concentração de vencimentos à frente enfrenta risco de refinanciamento. |
| `carteira_ativa` (janelas longas) | **Efeito safra**: expansão acelerada de crédito precede inadimplência em 12–24 meses — um dos achados mais consolidados da literatura de risco de crédito. |

## Convenção temporal

As novas variáveis seguem a mesma defasagem de publicação do restante do SCR (`lag_scr` em
`ml_metadata.json`): na origem `t` só se conhece o mês `t − lag`. O alvo continua em `t+6` e o
embargo é o mesmo dos notebooks 04 e 07 (horizonte + defasagem).

## Teste

A pergunta é objetiva: **as novas variáveis melhoram a detecção de eventos?** Compara-se o conjunto
base (as variáveis do notebook 07) com o estendido, sob protocolo idêntico e nas mesmas linhas,
usando DeLong e bootstrap de bloco mensal. Como teste secundário, verifica-se se o R² *within* da
regressão — negativo para todos os modelos até aqui — se torna positivo.

In [ ]:
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

BASE = Path().resolve().parent
PROC = BASE / 'data' / 'processed'
PARQ = PROC / 'scr_pj'
SEG_KEYS = ['cnae_secao', 'porte', 'modalidade']

with open(PROC / 'ml_metadata.json', encoding='utf-8') as f:
    meta = json.load(f)
FEATURE_COLS = meta['feature_cols']
SPLIT_DATE   = pd.Timestamp(meta['split_date'])
HORIZONTE    = int(meta.get('horizonte', 6))

# --- Defasagem de publicacao (revisao A1), gravada pelo notebook 03 ---
LAG_SCR = int(meta.get('lag_scr', 0))
NIVEL   = 'nivel_disp' if LAG_SCR else 'taxa_inadimplencia'
EMBARGO = HORIZONTE + LAG_SCR

COLS = ['data_base', 'cnae_secao', 'porte', 'modalidade', 'carteira_ativa',
        'ativo_problematico',
        'a_vencer_ate_90_dias', 'a_vencer_de_91_ate_360_dias',
        'a_vencer_de_361_ate_1080_dias', 'a_vencer_de_1081_ate_1800_dias',
        'a_vencer_de_1801_ate_5400_dias', 'a_vencer_acima_de_5400_dias']
SOMAR = [c for c in COLS if c not in ['data_base'] + SEG_KEYS]

# Agrega ARQUIVO A ARQUIVO (mesma pratica dos notebooks 01 e 02)
print('Agregando SCR com as colunas adicionais...')
parciais = []
for f in sorted(PARQ.glob('*.parquet')):
    if '2025' in f.name:          # ano incompleto, excluido como no notebook 03
        continue
    d = pd.read_parquet(f, columns=COLS)
    parciais.append(d.groupby(['data_base'] + SEG_KEYS, observed=True)[SOMAR].sum().reset_index())
    del d
    print(f'  {f.name} ok')

agg = pd.concat(parciais, ignore_index=True)
for c in SEG_KEYS:
    agg[c] = agg[c].str.replace('PJ - ', '', regex=False).str.strip()
agg = (agg.groupby(['data_base'] + SEG_KEYS, observed=True)[SOMAR].sum()
          .reset_index().sort_values(SEG_KEYS + ['data_base']).reset_index(drop=True))
print()
print(f'Agregado: {agg.shape[0]:,} linhas x {agg.shape[1]} colunas')
print(f'Defasagem de publicacao do SCR: {LAG_SCR} mes(es)  |  embargo: {EMBARGO} meses')

## 1. Construção das novas variáveis

In [ ]:
g = agg.groupby(SEG_KEYS, observed=True)
ca = agg['carteira_ativa'].replace(0, np.nan)

# --- A.1 Ativo problematico (indicador antecedente) ---
agg['problem_ratio']  = agg['ativo_problematico'] / ca
agg['problem_delta3'] = agg['problem_ratio'] - g['problem_ratio'].shift(3)
agg['problem_delta6'] = agg['problem_ratio'] - g['problem_ratio'].shift(6)

# --- A.2 Estrutura de maturidade ---
FAIXAS = {'a_vencer_ate_90_dias': 45, 'a_vencer_de_91_ate_360_dias': 225,
          'a_vencer_de_361_ate_1080_dias': 720, 'a_vencer_de_1081_ate_1800_dias': 1440,
          'a_vencer_de_1801_ate_5400_dias': 3600, 'a_vencer_acima_de_5400_dias': 7200}
total_venc = agg[list(FAIXAS)].sum(axis=1).replace(0, np.nan)

agg['venc90_share']  = agg['a_vencer_ate_90_dias'] / ca
agg['venc360_share'] = (agg['a_vencer_ate_90_dias']
                        + agg['a_vencer_de_91_ate_360_dias']) / ca
agg['longo_share']   = (agg['a_vencer_de_1081_ate_1800_dias']
                        + agg['a_vencer_de_1801_ate_5400_dias']
                        + agg['a_vencer_acima_de_5400_dias']) / ca
# Prazo medio ponderado, em meses
agg['duration_m'] = sum(agg[c] * d for c, d in FAIXAS.items()) / total_venc / 30.0
agg['venc360_delta3'] = agg['venc360_share'] - g['venc360_share'].shift(3)

# --- C.1 Efeito safra: crescimento da carteira em janelas longas ---
for k in (12, 18, 24):
    agg[f'carteira_growth{k}'] = agg['carteira_ativa'] / g['carteira_ativa'].shift(k) - 1

NOVAS = ['problem_ratio', 'problem_delta3', 'problem_delta6',
         'venc90_share', 'venc360_share', 'longo_share', 'duration_m', 'venc360_delta3',
         'carteira_growth12', 'carteira_growth18', 'carteira_growth24']

# --- Defasagem de publicacao (revisao A1) ---
# O que o modelo ve na origem t e o valor PUBLICADO, isto e, o de t - LAG_SCR. Todas as novas
# variaveis sao deslocadas dentro do segmento. `carteira_ativa` fica no valor real de t: ela
# NAO e variavel do modelo — e usada apenas para estratificar e ponderar a avaliacao (A3).
if LAG_SCR:
    g2 = agg.groupby(SEG_KEYS, observed=True)
    for c in NOVAS:
        agg[c] = g2[c].shift(LAG_SCR)

print(f'{len(NOVAS)} variaveis novas' + (f' (deslocadas {LAG_SCR} mes(es))' if LAG_SCR else '') + ':')
for c in NOVAS:
    print(f'  {c:20s} nulos: {agg[c].isna().mean():6.2%}  '
          f'mediana: {agg[c].median():+.4f}')

In [ ]:
# Junta ao dataset de ML preservando EXATAMENTE as mesmas linhas (inner join)
ml = (pd.read_parquet(PROC / 'ml_dataset.parquet')
        .sort_values('data_base').reset_index(drop=True))
ml = ml.drop(columns=[c for c in ['carteira_ativa'] if c in ml.columns])   # evita duplicata
n0 = len(ml)
df = ml.merge(agg[['data_base'] + SEG_KEYS + NOVAS + ['carteira_ativa']],
              on=['data_base'] + SEG_KEYS, how='left')
assert len(df) == n0, 'o merge alterou o numero de linhas'

# `problem_gap`: deterioracao que ainda NAO virou inadimplencia — ambos no ultimo valor publicado
df['problem_gap'] = df['problem_ratio'] - df[NIVEL]
NOVAS = NOVAS + ['problem_gap']

print(f'Linhas preservadas: {len(df):,} (identicas ao ml_dataset)')
print(f'Cobertura das novas variaveis: '
      f'{df[NOVAS].notna().all(axis=1).mean():.1%} das linhas sem nenhum nulo')

# --- Alvo de evento, identico ao notebook 07 (definido sobre a serie REAL) ---
df['delta6'] = df['target'] - df['taxa_inadimplencia']
embargo_lim = SPLIT_DATE - pd.DateOffset(months=EMBARGO)
mask_tr = df['data_base'] <  embargo_lim
mask_te = df['data_base'] >= SPLIT_DATE
LIMIAR = float(df.loc[mask_tr, 'delta6'].quantile(0.80))
df['evento'] = (df['delta6'] > LIMIAR).astype(int)

FEATS_BASE = FEATURE_COLS + [NIVEL]                        # 29, notebook 07
FEATS_EXT  = FEATS_BASE + NOVAS                           # estendido

tr, te = df[mask_tr], df[mask_te]
y_tr, y_te = tr['evento'].to_numpy(), te['evento'].to_numpy()

print()
print(f'Defasagem SCR: {LAG_SCR} mes(es)  |  embargo: {EMBARGO} meses  |  nivel conhecido: {NIVEL}')
print(f'Limiar: {LIMIAR*100:.2f} p.p.  |  eventos treino {y_tr.mean():.2%} / teste {y_te.mean():.2%}')
print(f'Conjunto BASE: {len(FEATS_BASE)} variaveis  |  ESTENDIDO: {len(FEATS_EXT)}')

## 2. Base vs. estendido — detecção de eventos

In [ ]:
def rf():
    return Pipeline([('imp', SimpleImputer(strategy='median')),
                     ('clf', RandomForestClassifier(n_estimators=300, max_depth=12,
                                                    min_samples_leaf=10, n_jobs=-1,
                                                    random_state=42))])

def xgb():
    return Pipeline([('imp', SimpleImputer(strategy='median')),
                     ('clf', XGBClassifier(n_estimators=400, learning_rate=0.05, max_depth=6,
                                           subsample=0.8, colsample_bytree=0.8,
                                           random_state=42, verbosity=0,
                                           eval_metric='logloss'))])

scores, linhas = {}, []
for nome_mod, fab in [('Random Forest', rf), ('XGBoost', xgb)]:
    for nome_cj, feats in [('base', FEATS_BASE), ('estendido', FEATS_EXT)]:
        m = fab().fit(tr[feats], y_tr)
        s = m.predict_proba(te[feats])[:, 1]
        scores[f'{nome_mod} ({nome_cj})'] = s
        top = np.argsort(-s)[:int(len(s) * 0.10)]
        linhas.append({'Modelo': nome_mod, 'Conjunto': nome_cj, 'n_vars': len(feats),
                       'AUC-ROC': round(roc_auc_score(y_te, s), 4),
                       'AUC-PR': round(average_precision_score(y_te, s), 4),
                       'P@10%': round(y_te[top].mean(), 4),
                       'Lift@10%': round(y_te[top].mean() / y_te.mean(), 2)})
        print(f'{nome_mod:14s} {nome_cj:10s} AUC={linhas[-1]["AUC-ROC"]:.4f}  '
              f'P@10%={linhas[-1]["P@10%"]:.3f}')

comp = pd.DataFrame(linhas)
print()
print(comp.to_string(index=False))
comp.to_csv(PROC / 'resultados_features_scr.csv', index=False)
print()
print('Salvo: resultados_features_scr.csv')

## 3. O ganho é significativo?

Mesma inferência do notebook 07: DeLong (assume independência) e bootstrap de bloco mensal
(preserva a dependência transversal do painel).

In [ ]:
from scipy import stats as sps

def _midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x); T = np.zeros(N); i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    T2 = np.empty(N); T2[J] = T
    return T2

def delong(y, s1, s2):
    ok = np.isfinite(s1) & np.isfinite(s2)
    y_, S = y[ok], np.vstack([s1[ok], s2[ok]])
    pos, neg = S[:, y_ == 1], S[:, y_ == 0]
    m, n = pos.shape[1], neg.shape[1]
    tx = np.array([_midrank(pos[r]) for r in range(2)])
    ty = np.array([_midrank(neg[r]) for r in range(2)])
    tz = np.array([_midrank(np.concatenate([pos[r], neg[r]])) for r in range(2)])
    aucs = (tz[:, :m].sum(axis=1) - m * (m + 1) / 2) / (m * n)
    Sig = np.cov((tz[:, :m] - tx) / n) / m + np.cov(1 - (tz[:, m:] - ty) / m) / n
    var = Sig[0, 0] + Sig[1, 1] - 2 * Sig[0, 1]
    z = (aucs[0] - aucs[1]) / np.sqrt(max(var, 1e-16))
    return aucs[0], aucs[1], 2 * (1 - sps.norm.cdf(abs(z)))

def boot_mensal(y, s1, s2, meses, B=600, seed=42):
    rng = np.random.default_rng(seed)
    uniq = np.unique(meses)
    idx_por_mes = {mm: np.where(meses == mm)[0] for mm in uniq}
    difs = []
    for _ in range(B):
        sel = rng.choice(uniq, size=len(uniq), replace=True)
        idx = np.concatenate([idx_por_mes[mm] for mm in sel])
        yy = y[idx]
        if yy.min() == yy.max():
            continue
        difs.append(roc_auc_score(yy, s1[idx]) - roc_auc_score(yy, s2[idx]))
    difs = np.array(difs)
    return np.percentile(difs, 2.5), np.percentile(difs, 97.5), (difs <= 0).mean()

meses_te = te['data_base'].to_numpy()
out = []
for mod in ['Random Forest', 'XGBoost']:
    a, b = scores[f'{mod} (estendido)'], scores[f'{mod} (base)']
    au_e, au_b, p = delong(y_te, a, b)
    lo, hi, pb = boot_mensal(y_te, a, b, meses_te)
    out.append({'Modelo': mod, 'AUC base': round(au_b, 4), 'AUC estendido': round(au_e, 4),
                'ganho': round(au_e - au_b, 4), 'p (DeLong)': round(p, 4),
                'IC95 boot': f'[{lo:+.4f}, {hi:+.4f}]', 'p (boot)': round(pb, 4)})

inf = pd.DataFrame(out)
print('=== Ganho do conjunto estendido sobre o base ===')
print(inf.to_string(index=False))
inf.to_csv(PROC / 'resultados_features_scr_inferencia.csv', index=False)
print()
print('Salvo: resultados_features_scr_inferencia.csv')

## 4. Quais das novas variáveis pesam?

In [ ]:
m = rf().fit(tr[FEATS_EXT], y_tr)
imp = (pd.Series(m.named_steps['clf'].feature_importances_, index=FEATS_EXT)
         .sort_values(ascending=False))
imp_novas = imp[imp.index.isin(NOVAS)]

print('Top 15 variaveis (conjunto estendido, Random Forest):')
for k, v in imp.head(15).items():
    marca = '  <-- NOVA' if k in NOVAS else ''
    print(f'  {k:24s} {v:.4f}{marca}')
print()
print(f'Peso somado das {len(NOVAS)} novas variaveis: {imp_novas.sum():.1%}')
print(f'Melhor nova variavel: {imp_novas.index[0]} ({imp_novas.iloc[0]:.4f})')

# Revisao C8: grava as importancias para que o artigo leia os numeros do CSV
_imp_df = imp.rename('importancia').reset_index().rename(columns={'index': 'variavel'})
_imp_df['nova'] = _imp_df['variavel'].isin(NOVAS)
_imp_df['peso_novas_total'] = float(imp_novas.sum())
_imp_df.to_csv(PROC / 'resultados_features_scr_importancias.csv', index=False)
print('Salvo: resultados_features_scr_importancias.csv')

## 5. Teste secundário — o R² *within* deixa de ser negativo?

O achado mais crítico dos notebooks anteriores foi o **R² intrassegmento negativo** para todos os
modelos: removido o nível médio de cada segmento, nenhum superava a média histórica do próprio
segmento. Se as novas variáveis realmente trazem informação temporal, esse número deve melhorar.

In [ ]:
def within_r2(y, pred, gid):
    seg_mean = pd.Series(y).groupby(gid).transform('mean').to_numpy()
    ss_within = float(np.sum((y - seg_mean) ** 2))
    return 1 - float(np.sum((y - pred) ** 2)) / ss_within

gid = te[SEG_KEYS].astype(str).agg('|'.join, axis=1).to_numpy()
y_tr_r, y_te_r = tr['target'].to_numpy(), te['target'].to_numpy()

print('R2 pooled e within (RandomForestRegressor, alvo = nivel em t+6):')
_w_rows = []
for nome_cj, feats in [('base', FEATS_BASE), ('estendido', FEATS_EXT)]:
    reg = Pipeline([('imp', SimpleImputer(strategy='median')),
                    ('reg', RandomForestRegressor(n_estimators=200, max_depth=12,
                                                  min_samples_leaf=10, n_jobs=-1,
                                                  random_state=42))]).fit(tr[feats], y_tr_r)
    p = reg.predict(te[feats])
    r2p = 1 - np.sum((y_te_r - p) ** 2) / np.sum((y_te_r - y_te_r.mean()) ** 2)
    print(f'  {nome_cj:10s} R2_pooled={r2p:+.4f}   R2_within={within_r2(y_te_r, p, gid):+.4f}   '
          f'RMSE={np.sqrt(np.mean((y_te_r - p) ** 2)):.4f}')
    _w_rows.append({'Conjunto': nome_cj, 'n_vars': len(feats), 'R2_pooled': round(r2p, 4),
                    'R2_within': round(within_r2(y_te_r, p, gid), 4),
                    'RMSE': round(float(np.sqrt(np.mean((y_te_r - p) ** 2))), 4)})
pd.DataFrame(_w_rows).to_csv(PROC / 'resultados_features_scr_within.csv', index=False)
print('Salvo: resultados_features_scr_within.csv')

## 7. Gravação do dataset estendido

Persiste o conjunto estendido para reuso pelos notebooks seguintes (em especial o `09`, que
executa o *backtest* de origem móvel), evitando repetir a agregação do SCR.

In [ ]:
# Colunas de identificacao e avaliacao (nao sao variaveis do modelo): chaves, serie real,
# alvos, carteira (para estratificar/ponderar — revisao A3), nivel publicado e volatilidade.
META = ['data_base'] + SEG_KEYS + ['taxa_inadimplencia', 'target', 'delta6', 'evento',
                                   'carteira_ativa']
for c in (NIVEL, 'vol12_disp'):
    if c in df.columns and c not in META:
        META.append(c)
out = df[META + [c for c in FEATS_EXT if c not in META]].copy()

destino = PROC / 'ml_dataset_ext.parquet'
out.to_parquet(destino, index=False)

meta_ext = {
    'feature_cols_base': FEATS_BASE,
    'feature_cols_ext':  FEATS_EXT,
    'novas':             NOVAS,
    'target_col':        'target',
    'evento_col':        'evento',
    'nivel_col':         NIVEL,
    'limiar_evento':     float(LIMIAR),
    'quantil_limiar':    0.80,
    'horizonte':         HORIZONTE,
    'lag_scr':           LAG_SCR,
    'embargo_meses':     EMBARGO,
    'lags_macro':        meta.get('lags_macro', {}),
    'split_date':        str(SPLIT_DATE.date()),
}
with open(PROC / 'ml_metadata_ext.json', 'w', encoding='utf-8') as f:
    json.dump(meta_ext, f, indent=2, ensure_ascii=False)

print(f'Salvo: {destino.name}  ({out.shape[0]:,} linhas x {out.shape[1]} colunas)')
print(f'Salvo: ml_metadata_ext.json  ({len(FEATS_EXT)} variaveis no conjunto estendido; '
      f'lag_scr={LAG_SCR}, embargo={EMBARGO})')